# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step example for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and its Croissant schema.

### Dataset Source
The dataset schema is provided via the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant if not already available
!pip install -q mlcroissant

## 1. Data Loading

Let's load metadata and records from the FAIR² dataset using `mlcroissant`. We will inspect the dataset summary first.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset and metadata
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata fields
metadata = dataset.metadata
print('Name:', metadata.name)
print('Description:', metadata.description)
print('Identifier:', metadata.identifier)
print('Published:', metadata.datePublished)
print('Version:', metadata.version)

## 2. Data Overview

Now, let's explore the record sets defined in the dataset. We'll list each available record set along with its `@id`, fields, and the IDs of those fields.

Using Croissant, every entity (record set, field, column, etc.) has a unique `@id`.

In [ ]:
# List all RecordSets and their fields (by @id), using mlcroissant's metadata

if hasattr(dataset.metadata, 'recordSet') and dataset.metadata.recordSet:
    record_sets = dataset.metadata.recordSet
else:
    # Fallback: try metadata._metadata['recordSet'] if recordSet is not set directly
    record_sets = dataset.metadata._metadata.get('recordSet', [])

print('Available record sets:')
record_set_ids = []
for rs in record_sets:
    rs_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None)
    rs_name = rs.get('name', '') if isinstance(rs, dict) else getattr(rs, 'name', '')
    print(f"- RecordSet @id: {rs_id}, name: {rs_name}")
    # Try to list fields
    fields = rs.get('field', []) if isinstance(rs, dict) else getattr(rs, 'field', [])
    if not fields:
        # Fallback: check singular 'fields'
        fields = rs.get('fields', []) if isinstance(rs, dict) else getattr(rs, 'fields', [])
    print(f"   Fields:")
    for f in fields:
        f_id = f['@id'] if isinstance(f, dict) else getattr(f, '@id', None)
        f_name = f.get('name', '') if isinstance(f, dict) else getattr(f, 'name', '')
        print(f"     - Field @id: {f_id}, name: {f_name}")
    record_set_ids.append(rs_id)

if not record_set_ids:
    print('No record sets found in metadata.')

## 3. Data Extraction

We will now extract data from the main record set(s) using their `@id`.

Each record set defines a logical table or set of records. We'll load each table as a pandas DataFrame and inspect the first few records and column names, referencing fields by their `@id`s.

In [ ]:
# Collect data from all record sets by their @id
dataframes = {}

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} rows from record set {rs_id}")
        else:
            print(f"No records found for {rs_id}")
    except Exception as e:
        print(f"Error loading records from {rs_id}: {e}")

if dataframes:
    # Pick the largest DataFrame as default for further analysis
    main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k]))
    print(f"\nDefaulting to main record set for preview: {main_record_set_id}")
    print("Columns:", dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print('No dataframes extracted.')

## 4. Exploratory Data Analysis (EDA)

Let's do some simple EDA. We'll:
- Identify a numeric field (column) by its `@id` and examine basic statistics.
- Filter for values in the numeric field above a threshold (e.g., age > 50 if available).
- Normalize that field.
- Group by a relevant categorical field by its `@id` (e.g., sex, MSI status, etc.).

**Note:** Adjust field `@id`s based on the actual columns shown above.

In [ ]:
# ---- EDA: numeric field analysis ----

# Use the main_record_set_id from previous extraction
df = dataframes[main_record_set_id]

# List candidate numeric fields
print('Available columns:')
for c in df.columns:
    print(f'- {c}')

# Select a numeric field by @id (replace with real @id if known; here we try to infer "Age" field by common names)
candidate_age_ids = [c for c in df.columns if 'age' in c.lower()]
if candidate_age_ids:
    numeric_field_id = candidate_age_ids[0]
else:
    # Fallback: pick a numeric-looking field
    numeric_field_id = df.select_dtypes(include='number').columns[0]
print(f"\nUsing numeric field @id: {numeric_field_id}")

# Filter: age > 50
threshold = 50
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field (by @id): look for a 'sex', 'status', or similar column
candidate_group_field_ids = [c for c in df.columns if any(x in c.lower() for x in ['sex', 'status', 'msi', 'anatomical'])]
if candidate_group_field_ids:
    group_field_id = candidate_group_field_ids[0]
    print(f"\nGrouping by field @id: {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df)
else:
    print('No suitable categorical field found for grouping.')

## 5. Visualization

Let's plot the distribution of the numeric field as well as means by group, using Matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Plot group means if available
if 'group_field_id' in locals():
    plt.figure(figsize=(7,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion

We have demonstrated how to load, inspect, and analyze the FAIR² dataset using its Croissant schema and the `mlcroissant` library.

Key steps include:
- Loading the dataset and metadata directly from its Croissant schema URL.
- Exploring available record sets, fields, and their unique `@id`s as defined by the schema.
- Extracting and working with the main data table as a DataFrame.
- Performing simple EDA: filtering, normalization, grouping, and plotting.

For in-depth study:
- Consult the exact schema and documentation for definitions of each `@id`.
- Explore the fields for clinical meaning and more advanced analyses.

For more: https://mlcommons.github.io/croissant/